# Object Detection using YOLOv8 Transfer Learning

**Image Processing Project · Phase 2 — Intermediate · Object Detection**

**Framework:** PyTorch + Ultralytics YOLOv8  
**Dataset:** PASCAL VOC 2012  
**GPU:** Kaggle T4  
**Deployment:** Gradio + ONNX + FastAPI

## Project Objective

This project builds a YOLOv8 object detection model fine-tuned on the
PASCAL VOC 2012 dataset.

The model will simultaneously:

- Localise multiple objects in a single image using bounding boxes
- Classify each detected object into one of 20 VOC classes
- Provide a confidence score for each detection

## YOLOv8 Motivation

YOLOv8 is a one-stage object detection model that performs localisation
and classification in a single forward pass.

In this project, YOLOv8-nano pretrained on COCO will be fine-tuned on
the 20 PASCAL VOC object classes.

## Real-World Applications

- Autonomous driving
- Industrial quality control
- Retail analytics
- Security and surveillance
- Medical imaging
- Agriculture

## Hugging Face Spaces

A Gradio-based demo will be deployed on Hugging Face Spaces after
training and evaluation.

In [ ]:
# Step 2: Install and Imports

!pip install -q ultralytics

from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image
import torch
import os

# GPU Check
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

In [ ]:
# Step 3: Explore Raw Dataset

# Dataset paths

images_path = "/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val/JPEGImages"

annotations_path = "/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val/Annotations"

print("Images path:", images_path)
print("Annotations path:", annotations_path)

print("\nNumber of images:", len(os.listdir(images_path)))
print("Number of annotations:", len(os.listdir(annotations_path)))

In [ ]:
# Step 3: Explore Raw Dataset
# Check 3 images and their XML annotation files

# Get first 3 image files
image_files = sorted(os.listdir(images_path))[:3]

print("First 3 images:")

for image_file in image_files:
    print(image_file)

print("\nCorresponding XML annotations:")

for image_file in image_files:
    xml_file = os.path.splitext(image_file)[0] + ".xml"
    print(xml_file)

In [ ]:
# Step 3: Explore Raw Dataset
# Read XML annotation

import xml.etree.ElementTree as ET

xml_file = os.path.join(annotations_path, "2007_000027.xml")

tree = ET.parse(xml_file)
root = tree.getroot()

print("Objects in 2007_000027.xml:")

for obj in root.findall("object"):
    class_name = obj.find("name").text

    bbox = obj.find("bndbox")

    xmin = bbox.find("xmin").text
    ymin = bbox.find("ymin").text
    xmax = bbox.find("xmax").text
    ymax = bbox.find("ymax").text

    print("Class:", class_name)
    print("Bounding Box:", xmin, ymin, xmax, ymax)
    print()

In [ ]:
# Step 3: Explore Raw Dataset
# Print all 20 PASCAL VOC classes

VOC_CLASSES = [
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "pottedplant",
    "sheep",
    "sofa",
    "train",
    "tvmonitor"
]

print("PASCAL VOC 2012 Classes:")

for i, class_name in enumerate(VOC_CLASSES):
    print(i, ":", class_name)

In [ ]:
# Step 4: EDA - Class Distribution

import xml.etree.ElementTree as ET
from collections import Counter
import matplotlib.pyplot as plt

# Count objects in all XML annotation files
class_counts = Counter()

xml_files = [
    f for f in os.listdir(annotations_path)
    if f.endswith(".xml")
]

for xml_file in xml_files:
    xml_path = os.path.join(annotations_path, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall("object"):
        class_name = obj.find("name").text
        class_counts[class_name] += 1

print("Number of objects in each class:")

for class_name in VOC_CLASSES:
    print(class_name, ":", class_counts[class_name])

In [ ]:
# Step 4: EDA - Class Distribution
# Plot number of objects per class

counts = [class_counts[class_name] for class_name in VOC_CLASSES]

plt.figure(figsize=(12, 6))

plt.bar(VOC_CLASSES, counts)

plt.xlabel("Object Class")
plt.ylabel("Number of Objects")
plt.title("PASCAL VOC 2012 - Class Distribution")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Step 4: EDA - Objects per Image

objects_per_image = []

for xml_file in xml_files:
    xml_path = os.path.join(annotations_path, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    num_objects = len(root.findall("object"))
    objects_per_image.append(num_objects)

# Histogram
plt.figure(figsize=(10, 6))

plt.hist(
    objects_per_image,
    bins=range(1, max(objects_per_image) + 2)
)

plt.xlabel("Number of Objects in an Image")
plt.ylabel("Number of Images")
plt.title("PASCAL VOC 2012 - Objects per Image")

plt.tight_layout()
plt.show()

In [ ]:
# Step 4: EDA - Bounding Box Size Scatter

bbox_widths = []
bbox_heights = []

for xml_file in xml_files:
    xml_path = os.path.join(annotations_path, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall("object"):
        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        bbox_widths.append(xmax - xmin)
        bbox_heights.append(ymax - ymin)

# Scatter plot
plt.figure(figsize=(10, 6))

plt.scatter(bbox_widths, bbox_heights, alpha=0.3)

plt.xlabel("Bounding Box Width")
plt.ylabel("Bounding Box Height")
plt.title("PASCAL VOC 2012 - Bounding Box Size Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Dataset Preparation — YOLO Format

VOC_CLASSES = [
    "aeroplane","bicycle","bird","boat","bottle","bus","car","cat",
    "chair","cow","diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor"
]

class_to_id = {
    class_name: i
    for i, class_name in enumerate(VOC_CLASSES)
}

print("YOLO Class ID Mapping:")
for class_name, class_id in class_to_id.items():
    print(class_id, ":", class_name)

In [ ]:
import xml.etree.ElementTree as ET

sample_xml = os.path.join(
    annotations_path,
    "2007_000027.xml"
)

tree = ET.parse(sample_xml)
root = tree.getroot()

print("Image:", root.find("filename").text)

print("\nObjects:")

for obj in root.findall("object"):
    class_name = obj.find("name").text
    bbox = obj.find("bndbox")

    xmin = float(bbox.find("xmin").text)
    ymin = float(bbox.find("ymin").text)
    xmax = float(bbox.find("xmax").text)
    ymax = float(bbox.find("ymax").text)

    print("Class:", class_name)
    print("Bounding Box:", xmin, ymin, xmax, ymax)

In [ ]:
# Get image dimensions
image_width = int(root.find("size/width").text)
image_height = int(root.find("size/height").text)

# Get class ID
class_name = root.find("object/name").text
class_id = class_to_id[class_name]

# Get bounding box
bbox = root.find("object/bndbox")

xmin = float(bbox.find("xmin").text)
ymin = float(bbox.find("ymin").text)
xmax = float(bbox.find("xmax").text)
ymax = float(bbox.find("ymax").text)

# Convert to YOLO format
x_center = (xmin + xmax) / 2
y_center = (ymin + ymax) / 2
bbox_width = xmax - xmin
bbox_height = ymax - ymin

# Normalize coordinates
x_center /= image_width
y_center /= image_height
bbox_width /= image_width
bbox_height /= image_height

print("Image Width:", image_width)
print("Image Height:", image_height)
print("Class ID:", class_id)

print("\nYOLO Format:")
print(
    class_id,
    x_center,
    y_center,
    bbox_width,
    bbox_height
)

In [ ]:
# Create YOLO dataset folder structure

yolo_dataset_path = "/kaggle/working/voc_yolo"

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    os.makedirs(
        os.path.join(yolo_dataset_path, folder),
        exist_ok=True
    )

print("YOLO dataset folders created successfully!")
print("Dataset path:", yolo_dataset_path)

for folder in folders:
    print(os.path.join(yolo_dataset_path, folder))

In [ ]:
from sklearn.model_selection import train_test_split

image_files = [
    f for f in os.listdir(images_path)
    if f.lower().endswith(".jpg")
]

train_images, val_images = train_test_split(
    image_files,
    test_size=0.20,
    random_state=42
)

print("Total images:", len(image_files))
print("Training images:", len(train_images))
print("Validation images:", len(val_images))

In [ ]:
import shutil

# Copy training images
for image_file in train_images:
    source = os.path.join(images_path, image_file)
    destination = os.path.join(
        yolo_dataset_path,
        "images/train",
        image_file
    )
    shutil.copy2(source, destination)

# Copy validation images
for image_file in val_images:
    source = os.path.join(images_path, image_file)
    destination = os.path.join(
        yolo_dataset_path,
        "images/val",
        image_file
    )
    shutil.copy2(source, destination)

print("Images copied successfully!")
print(
    "Train images:",
    len(os.listdir(os.path.join(yolo_dataset_path, "images/train")))
)
print(
    "Validation images:",
    len(os.listdir(os.path.join(yolo_dataset_path, "images/val")))
)

In [ ]:
def convert_xml_to_yolo(xml_file, output_txt):

    xml_path = os.path.join(annotations_path, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    image_width = int(root.find("size/width").text)
    image_height = int(root.find("size/height").text)

    yolo_labels = []

    for obj in root.findall("object"):

        class_name = obj.find("name").text
        class_id = class_to_id[class_name]

        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2

        bbox_width = xmax - xmin
        bbox_height = ymax - ymin

        # Normalize coordinates
        x_center /= image_width
        y_center /= image_height
        bbox_width /= image_width
        bbox_height /= image_height

        yolo_labels.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} "
            f"{bbox_width:.6f} {bbox_height:.6f}"
        )

    with open(output_txt, "w") as f:
        f.write("\n".join(yolo_labels))

print("Conversion function is ready!")

In [ ]:
# Step 5: Dataset Preparation — YOLO Format
# Convert XML Annotations to YOLO TXT Labels

# Convert training labels
for image_file in train_images:

    xml_file = os.path.splitext(image_file)[0] + ".xml"

    output_txt = os.path.join(
        yolo_dataset_path,
        "labels/train",
        os.path.splitext(image_file)[0] + ".txt"
    )

    convert_xml_to_yolo(xml_file, output_txt)


# Convert validation labels
for image_file in val_images:

    xml_file = os.path.splitext(image_file)[0] + ".xml"

    output_txt = os.path.join(
        yolo_dataset_path,
        "labels/val",
        os.path.splitext(image_file)[0] + ".txt"
    )

    convert_xml_to_yolo(xml_file, output_txt)


print("YOLO label files created successfully!")
print(
    "Training labels:",
    len(os.listdir(os.path.join(yolo_dataset_path, "labels/train")))
)
print(
    "Validation labels:",
    len(os.listdir(os.path.join(yolo_dataset_path, "labels/val")))
)

In [ ]:
# Step 5: Dataset Preparation — YOLO Format
# Create data.yaml

yaml_content = f"""
path: {yolo_dataset_path}

train: images/train
val: images/val

nc: 20

names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor
"""

yaml_path = os.path.join(yolo_dataset_path, "data.yaml")

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")
print("Location:", yaml_path)

In [ ]:
# Step 5: Dataset Preparation — YOLO Format
# Final Verification

print("YOLO Dataset Verification")
print("-------------------------")

print("Train images:",
      len(os.listdir(os.path.join(yolo_dataset_path, "images/train"))))

print("Val images:",
      len(os.listdir(os.path.join(yolo_dataset_path, "images/val"))))

print("Train labels:",
      len(os.listdir(os.path.join(yolo_dataset_path, "labels/train"))))

print("Val labels:",
      len(os.listdir(os.path.join(yolo_dataset_path, "labels/val"))))

print("data.yaml exists:",
      os.path.exists(yaml_path))

In [ ]:
# Step 6: Convert to YOLO Format
# Visual verification of YOLO labels

import random
import cv2
import matplotlib.pyplot as plt

# Select 3 validation images
sample_images = random.sample(val_images, 3)

for image_file in sample_images:

    # Image path
    image_path = os.path.join(
        yolo_dataset_path,
        "images/val",
        image_file
    )

    # Label path
    label_path = os.path.join(
        yolo_dataset_path,
        "labels/val",
        os.path.splitext(image_file)[0] + ".txt"
    )

    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image_height, image_width = image.shape[:2]

    # Read YOLO labels
    with open(label_path, "r") as f:
        labels = f.readlines()

    # Draw bounding boxes
    for label in labels:

        class_id, x_center, y_center, width, height = map(
            float, label.strip().split()
        )

        x_center *= image_width
        y_center *= image_height
        width *= image_width
        height *= image_height

        xmin = int(x_center - width / 2)
        ymin = int(y_center - height / 2)
        xmax = int(x_center + width / 2)
        ymax = int(y_center + height / 2)

        cv2.rectangle(
            image,
            (xmin, ymin),
            (xmax, ymax),
            (255, 0, 0),
            2
        )

        class_name = VOC_CLASSES[int(class_id)]

        cv2.putText(
            image,
            class_name,
            (xmin, max(ymin - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    # Display image
    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.title(image_file)
    plt.axis("off")
    plt.show()

In [ ]:
# Load YOLOv8 model

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

print("YOLOv8 model loaded successfully!")

In [ ]:
# Step 8: Fine-tune on VOC

results = model.train(
    data=yaml_path,
    epochs=50,
    batch=16,
    imgsz=640,
    device=0
)

In [ ]:
# Step 9: Training Curves

import pandas as pd
import matplotlib.pyplot as plt

results_csv = "/kaggle/working/runs/detect/train/results.csv"

results_df = pd.read_csv(results_csv)

print("Results CSV loaded successfully!")
print("Number of epochs:", len(results_df))

print("\nAvailable metrics:")
print(results_df.columns.tolist())

In [ ]:
# Step 9: Training Curves - Loss

plt.figure(figsize=(10, 6))

plt.plot(
    results_df["epoch"],
    results_df["train/box_loss"],
    label="Training Box Loss"
)

plt.plot(
    results_df["epoch"],
    results_df["train/cls_loss"],
    label="Training Classification Loss"
)

plt.plot(
    results_df["epoch"],
    results_df["train/dfl_loss"],
    label="Training DFL Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("YOLOv8 Training Loss Curves")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Step 9: Training Curves - mAP

plt.figure(figsize=(10, 6))

plt.plot(
    results_df["epoch"],
    results_df["metrics/mAP50(B)"],
    label="mAP@50"
)

plt.plot(
    results_df["epoch"],
    results_df["metrics/mAP50-95(B)"],
    label="mAP@50-95"
)

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("YOLOv8 Validation mAP Curves")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Step 10: Model Evaluation

best_model_path = "/kaggle/working/runs/detect/train/weights/best.pt"

best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=yaml_path,
    imgsz=640,
    device=0
)

print("\nEvaluation Results")
print("------------------")
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

In [ ]:
# Step 11: Detection Grid

import matplotlib.pyplot as plt
import random

# Select 12 random validation images
sample_images = random.sample(val_images, 12)

# Run predictions
prediction_results = best_model.predict(
    source=[
        os.path.join(yolo_dataset_path, "images/val", img)
        for img in sample_images
    ],
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)

print("Predictions completed for 12 images!")

In [ ]:
# Step 11: Detection Grid - Display Predictions

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax, result, image_file in zip(
    axes.flatten(),
    prediction_results,
    sample_images
):
    plotted_image = result.plot()

    ax.imshow(
        plotted_image[:, :, ::-1]
    )

    ax.set_title(image_file)
    ax.axis("off")

plt.suptitle(
    "YOLOv8 Detection Grid",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 12: NMS Threshold Experiment

confidence_thresholds = [0.10, 0.25, 0.50, 0.75]

test_image = os.path.join(
    yolo_dataset_path,
    "images/val",
    sample_images[0]
)

print("Test image:", sample_images[0])
print("Confidence thresholds:", confidence_thresholds)

In [ ]:
# Step 12: NMS Threshold Experiment - Generate Predictions

nms_results = {}

for conf_threshold in confidence_thresholds:

    result = best_model.predict(
        source=test_image,
        imgsz=640,
        conf=conf_threshold,
        device=0,
        verbose=False
    )[0]

    nms_results[conf_threshold] = result

print("Predictions generated for all confidence thresholds!")

In [ ]:
# Step 12: NMS Threshold Experiment - Visual Comparison

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, conf_threshold in zip(axes, confidence_thresholds):

    result = nms_results[conf_threshold]

    plotted_image = result.plot()

    ax.imshow(plotted_image[:, :, ::-1])
    ax.set_title(f"Confidence = {conf_threshold}")
    ax.axis("off")

plt.suptitle(
    "NMS Confidence Threshold Comparison",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 13: Per-Class AP

class_names = VOC_CLASSES
class_ap50 = metrics.box.ap50

print("Per-Class AP@50")
print("----------------")

for class_name, ap in zip(class_names, class_ap50):
    print(f"{class_name}: {ap:.3f}")

In [ ]:
# Step 13: Per-Class AP@50 Bar Chart

plt.figure(figsize=(14, 8))

plt.bar(
    class_names,
    class_ap50
)

plt.xlabel("VOC Classes")
plt.ylabel("AP@50")
plt.title("Per-Class AP@50 - YOLOv8")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Step 14: Error Analysis

error_image = sample_images[0]

error_image_path = os.path.join(
    yolo_dataset_path,
    "images/val",
    error_image
)

error_result = best_model.predict(
    source=error_image_path,
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)[0]

print("Error analysis image:", error_image)
print("Prediction completed!")

In [ ]:
# Step 14: Error Analysis - Ground Truth vs Prediction

import xml.etree.ElementTree as ET

# Get image information
image_path = error_image_path
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

image_height, image_width = image.shape[:2]

# Read original XML annotation
xml_file = os.path.splitext(error_image)[0] + ".xml"
xml_path = os.path.join(annotations_path, xml_file)

tree = ET.parse(xml_path)
root = tree.getroot()

# Create Ground Truth image
gt_image = image.copy()

for obj in root.findall("object"):

    class_name = obj.find("name").text
    bbox = obj.find("bndbox")

    xmin = int(float(bbox.find("xmin").text))
    ymin = int(float(bbox.find("ymin").text))
    xmax = int(float(bbox.find("xmax").text))
    ymax = int(float(bbox.find("ymax").text))

    cv2.rectangle(
        gt_image,
        (xmin, ymin),
        (xmax, ymax),
        (255, 0, 0),
        2
    )

    cv2.putText(
        gt_image,
        class_name,
        (xmin, max(ymin - 5, 15)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 0, 0),
        2
    )

# Create Prediction image
pred_image = error_result.plot()
pred_image = pred_image[:, :, ::-1]

# Display side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(gt_image)
axes[0].set_title("Ground Truth")
axes[0].axis("off")

axes[1].imshow(pred_image)
axes[1].set_title("YOLOv8 Prediction")
axes[1].axis("off")

plt.suptitle(
    "Ground Truth vs YOLOv8 Prediction",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 14: Error Analysis - 3 Scenes

error_samples = sample_images[:3]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for row, image_file in enumerate(error_samples):

    image_path = os.path.join(
        yolo_dataset_path,
        "images/val",
        image_file
    )

    # Original image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Ground Truth
    gt_image = image.copy()

    xml_file = os.path.splitext(image_file)[0] + ".xml"
    xml_path = os.path.join(annotations_path, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall("object"):

        class_name = obj.find("name").text
        bbox = obj.find("bndbox")

        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))

        cv2.rectangle(
            gt_image,
            (xmin, ymin),
            (xmax, ymax),
            (255, 0, 0),
            2
        )

        cv2.putText(
            gt_image,
            class_name,
            (xmin, max(ymin - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    # Prediction
    result = best_model.predict(
        source=image_path,
        imgsz=640,
        conf=0.25,
        device=0,
        verbose=False
    )[0]

    pred_image = result.plot()
    pred_image = pred_image[:, :, ::-1]

    # Display
    axes[row, 0].imshow(gt_image)
    axes[row, 0].set_title(f"Ground Truth - {image_file}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(pred_image)
    axes[row, 1].set_title(f"YOLOv8 Prediction - {image_file}")
    axes[row, 1].axis("off")

plt.suptitle(
    "Ground Truth vs YOLOv8 Predictions - 3 Scenes",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 15: Precision-Recall Curve

import os
from IPython.display import Image as IPImage, display

pr_curve_path = "/kaggle/working/runs/detect/val/BoxPR_curve.png"

if os.path.exists(pr_curve_path):
    print("PR curve found!")
    display(IPImage(filename=pr_curve_path))
else:
    print("PR curve not found. Generating validation plots...")

    best_model.val(
        data=yaml_path,
        imgsz=640,
        device=0,
        plots=True
    )

# Step 16: Conclusions

## Project Summary

This project implemented object detection using YOLOv8 transfer learning on the PASCAL VOC 2012 dataset.

The dataset contains 17,125 images with 20 object classes. The original PASCAL VOC XML annotations were successfully converted into YOLO format with normalized bounding box coordinates.

A pretrained YOLOv8n model was fine-tuned for 50 epochs using a Tesla T4 GPU.

## Final Evaluation Results

- Precision: 70.04%
- Recall: 57.43%
- mAP@50: 64.46%
- mAP@50-95: 47.84%

## Analysis

The model successfully learned to detect multiple object categories and produced bounding boxes for objects in validation images.

The detection grid and Ground Truth vs Prediction comparisons were used for visual verification.

The NMS confidence threshold experiment showed how changing the confidence threshold affects the number of displayed detections.

Per-class AP analysis showed that detection performance varies across the 20 VOC classes.

## Limitations

Some object classes have lower AP than others. This can occur because of differences in object appearance, object size, background complexity, occlusion, and the number of training examples.

The overall mAP@50 was 64.46%, which is slightly below the project's target range of 0.65–0.80.

## Future Improvements

- Train for more epochs.
- Experiment with larger YOLOv8 models.
- Apply additional data augmentation.
- Tune confidence and NMS thresholds.
- Increase or improve training data.
- Perform further hyperparameter tuning.

## Final Conclusion

The project successfully demonstrated YOLOv8 transfer learning for multi-class object detection on PASCAL VOC 2012. The trained model can detect objects and predict their locations using bounding boxes. Evaluation metrics, detection visualizations, per-class AP, NMS experiments, and error analysis were used to assess the model's performance.

In [ ]:
# Step 17: Export Best Model to ONNX

onnx_path = best_model.export(
    format="onnx",
    imgsz=640
)

print("ONNX model exported successfully!")
print("Path:", onnx_path)

In [ ]:
# Step 18: Verify ONNX Model

import os

onnx_file = "/kaggle/working/runs/detect/train/weights/best.onnx"

if os.path.exists(onnx_file):
    file_size = os.path.getsize(onnx_file) / (1024 * 1024)

    print("ONNX model found successfully!")
    print("Path:", onnx_file)
    print(f"File size: {file_size:.2f} MB")
else:
    print("ONNX model not found!")

In [ ]:
# Step 19: Gradio App

!pip install -q gradio

import gradio as gr

def predict_image(image):
    results = best_model.predict(
        source=image,
        imgsz=640,
        conf=0.25,
        device=0,
        verbose=False
    )

    result = results[0]

    # Image with predicted bounding boxes
    output_image = result.plot()

    # Convert BGR to RGB
    output_image = output_image[:, :, ::-1]

    return output_image

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Image(type="numpy"),
    title="YOLOv8 Object Detection",
    description="Upload an image and YOLOv8 will detect objects from the 20 PASCAL VOC classes."
)

demo.launch(share=True)